### Functional Manner to Compute Results Table

In [1]:
import pandas as pd
import warnings
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
def prep_event_data(cme_df, pm_df, kalshi_df=None):
    # Standardize CME (Using prob_no_change)
    df_cme = cme_df[["time", "prob_no_change"]].copy()
    df_cme["time"] = pd.to_datetime(df_cme["time"])
    df_cme = df_cme.rename(columns={"time": "timestamp"}).set_index("timestamp")
    cme_min = df_cme.resample("1Min").last()

    # Standardize Polymarket
    df_pm = pm_df[["datetime", "yes_price"]].copy()
    df_pm["datetime"] = pd.to_datetime(df_pm["datetime"])
    df_pm = df_pm.rename(columns={"datetime": "timestamp", "yes_price": "yes_price_PM"}).set_index("timestamp")
    pm_min = df_pm.resample("1Min").last().shift(1)

    # Merge Base Data
    merged_df = pd.merge(cme_min, pm_min, left_index=True, right_index=True, how="outer")

    # Add Kalshi
    if kalshi_df is not None:
        df_k = kalshi_df[["created_time", "yes_price"]].copy()
        df_k["created_time"] = pd.to_datetime(df_k["created_time"], format='mixed')
        df_k = df_k.rename(columns={"created_time": "timestamp", "yes_price": "yes_price_KAL"}).set_index("timestamp")
        kalshi_min = df_k.resample("1Min").last().shift(1)
        merged_df = pd.merge(merged_df, kalshi_min, left_index=True, right_index=True, how="outer")

    # Clean & Fill
    merged_df = merged_df.sort_index().ffill().dropna()

    # Alive Zone Filtering (Updated to prob_no_change)
    window_mins = 3 * 24 * 60
    rolling_var = merged_df["prob_no_change"].rolling(window=window_mins, min_periods=window_mins).var()
    is_active = rolling_var > 1e-6
    
    if is_active.any():
        first_active_t = merged_df[is_active].index.min()
        start_t = max(merged_df.index.min(), first_active_t - pd.Timedelta(minutes=window_mins))
        merged_df = merged_df.loc[start_t:]
    
    return merged_df

In [3]:
def plot_event_probabilities(df_clean, event_name="Event"):
    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax1.set_xlabel("Timestamp", fontsize=12, labelpad=10)
    ax1.set_ylabel("Implied Probability", fontsize=12)

    # Plotting CME's prob_no_change
    ax1.plot(df_clean.index, df_clean["prob_no_change"], color="#1f77b4", linewidth=1.5, label="CME")
    
    if "yes_price_PM" in df_clean.columns:
        ax1.plot(df_clean.index, df_clean["yes_price_PM"], color="#ff7f0e", linewidth=1.5, label="Polymarket")
    if "yes_price_KAL" in df_clean.columns:
        ax1.plot(df_clean.index, df_clean["yes_price_KAL"], color="#2ca02c", linewidth=1.5, label="Kalshi")

    ax1.grid(True, linestyle="--", alpha=0.5)
    ax1.legend(loc="upper left", fontsize=10, framealpha = 0.8)
    #plt.suptitle(f"{event_name}", fontsize=14, fontweight="bold", y=0.96)
    plt.show()

In [4]:
def run_granger_suite(df_clean, event_date, contract_type, mkt_1="CME", mkt_2="PM", lags=[1, 5, 30, 60]):
    """
    Runs Granger Causality for all specified lags in BOTH directions.
    Extracts the F-test (for Granger significance) and the OLS coefficient/p-value 
    (for magnitude and direction) for each relationship.
    """
    results_list = []
    
    # Map names to actual DataFrame columns
    col_map = {
        "CME": "prob_no_change",
        "PM": "yes_price_PM",
        "KAL": "yes_price_KAL"
    }
    
    col_1 = col_map.get(mkt_1)
    col_2 = col_map.get(mkt_2)
    
    # Pre-calculate the difference series
    test_df = pd.DataFrame()
    for lag in lags:
        test_df[f"{mkt_1}_diff_{lag}"] = df_clean[col_1].diff().rolling(window=lag).sum()
        test_df[f"{mkt_2}_diff_{lag}"] = df_clean[col_2].diff().rolling(window=lag).sum()
        
    test_df = test_df.dropna()
    
    # Run the tests for each lag
    for lag in lags:
        # Define the column names for this lag
        src_col = f"{mkt_1}_diff_{lag}"
        tgt_col = f"{mkt_2}_diff_{lag}"
        
        # Test Direction 1: Mkt 1 causes Mkt 2 (CME -> PM)
        # Statsmodels Granger function returns a dict where key '1' corresponds to the lag tested
        res_dir1 = grangercausalitytests(test_df[[tgt_col, src_col]], maxlag=1, verbose=False)
        
        # Extract F-test results
        f_stat_1 = res_dir1[1][0]['ssr_ftest'][0]
        p_val_1  = res_dir1[1][0]['ssr_ftest'][1]
        
        # Extract OLS regression results for the coefficient
        ols_res_1 = res_dir1[1][1][1]
        coef_1 = ols_res_1.params[1] 
        p_t_1  = ols_res_1.pvalues[1]
        
        results_list.append({
            "event_date": event_date,
            "contract": contract_type,
            "source_market": mkt_1,
            "target_market": mkt_2,
            "lag_minutes": lag,
            "f_stat": round(f_stat_1, 4),
            "p_value": round(p_val_1, 6),
            "coef": round(coef_1, 4),
            "coef_p_value": round(p_t_1, 6),
            "is_sig_5pct": p_val_1 < 0.05
        })
        
        # Test Direction 2: Mkt 2 causes Mkt 1 (PM -> CME)
        res_dir2 = grangercausalitytests(test_df[[src_col, tgt_col]], maxlag=1, verbose=False)
        f_stat_2 = res_dir2[1][0]['ssr_ftest'][0]
        p_val_2  = res_dir2[1][0]['ssr_ftest'][1]
        
        # Extract OLS regression results for the coefficient
        ols_res_2 = res_dir2[1][1][1]
        coef_2 = ols_res_2.params[1]
        p_t_2  = ols_res_2.pvalues[1]
        
        results_list.append({
            "event_date": event_date,
            "contract": contract_type,
            "source_market": mkt_2,
            "target_market": mkt_1,
            "lag_minutes": lag,
            "f_stat": round(f_stat_2, 4),
            "p_value": round(p_val_2, 6),
            "coef": round(coef_2, 4),
            "coef_p_value": round(p_t_2, 6),
            "is_sig_5pct": p_val_2 < 0.05
        })

    return results_list

### Granger Causality Results for All Events for 2025 (0bp cut Market)

In [ ]:
# Suppress warnings to keep the console output clean
warnings.simplefilter(action='ignore', category=FutureWarning)

# Configuration (2025 Events)
event_config = {
    "2025-01-29": {"month_code": "JAN"},
    "2025-03-19": {"month_code": "MAR"},
    "2025-05-07": {"month_code": "MAY"},
    "2025-06-18": {"month_code": "JUN"},
    "2025-07-30": {"month_code": "JUL"},
    "2025-09-17": {"month_code": "SEP"},
    "2025-10-29": {"month_code": "OCT"},
    "2025-12-10": {"month_code": "DEC"}
}

market_pairs = [("CME", "PM"), ("CME", "KAL"), ("PM", "KAL")]

# Main Processing Loop
all_master_results = []

for event_date, config in event_config.items():
    month = config["month_code"]
    print(f"\n{'#'*60}\nPROCESSING: {event_date} ({month})\n{'#'*60}")
    
    try:
        # Load files (Updated paths for 0bp)
        cme_raw = pd.read_csv(f"../../data/processed/CME_implied_probabilities/CME_IMP_{month}_2025.csv")
        pm_raw = pd.read_csv(f"../../data/raw/Polymarket/data_0bp_cut/PM_{month}_2025.csv")
        kal_raw = pd.read_csv(f"../../data/raw/Kalshi/data_0bp_cut/kalshi_{month}_2025.csv")
        
        # Prep data
        clean_df = prep_event_data(cme_df=cme_raw, pm_df=pm_raw, kalshi_df=kal_raw)
        
        # Visualization (optional)
        # print(f"  Plotting probabilities for {event_date}...")
        # plot_event_probabilities(df_clean=clean_df, event_name=f"FOMC {event_date} (0bp_dec)")
        
        # Run suites
        for mkt_a, mkt_b in market_pairs:
            print(f"  Testing {mkt_a} -> {mkt_b}...")
            pair_results = run_granger_suite(
                df_clean=clean_df, 
                event_date=event_date, 
                contract_type="0bp", 
                mkt_1=mkt_a, 
                mkt_2=mkt_b,
                lags=[1, 5, 30, 60]
            )
            all_master_results.extend(pair_results)
            
    except Exception as e:
        print(f"X Error processing {event_date}: {e}")


############################################################
PROCESSING: 2025-01-29 (JAN)
############################################################
  Testing CME -> PM...
  Testing CME -> KAL...
  Testing PM -> KAL...

############################################################
PROCESSING: 2025-03-19 (MAR)
############################################################
  Testing CME -> PM...
  Testing CME -> KAL...
  Testing PM -> KAL...

############################################################
PROCESSING: 2025-05-07 (MAY)
############################################################
  Testing CME -> PM...
  Testing CME -> KAL...
  Testing PM -> KAL...

############################################################
PROCESSING: 2025-06-18 (JUN)
############################################################
  Testing CME -> PM...
  Testing CME -> KAL...
  Testing PM -> KAL...

############################################################
PROCESSING: 2025-07-30 (JUL)
#####################

In [6]:
# Viewing the results as pandas dataframe 
final_results_df = pd.DataFrame(all_master_results)
final_results_df

,event_date,contract,source_market,target_market,lag_minutes,f_stat,p_value,coef,coef_p_value,is_sig_5pct
0,2025-01-29,0bp,CME,PM,1,4.8634,0.027436,0.0140,0.027436,True
1,2025-01-29,0bp,PM,CME,1,5.7082,0.016889,-0.0060,0.016889,True
2,2025-01-29,0bp,CME,PM,5,12.0456,0.000520,0.0150,0.000520,True
3,2025-01-29,0bp,PM,CME,5,0.0677,0.794759,0.0005,0.794759,False
4,2025-01-29,0bp,CME,PM,30,81.3562,0.000000,0.0197,0.000000,True
...,...,...,...,...,...,...,...,...,...,...
187,2025-12-10,0bp,KAL,PM,5,183.0988,0.000000,0.0259,0.000000,True
188,2025-12-10,0bp,PM,KAL,30,1402.8083,0.000000,0.0314,0.000000,True
189,2025-12-10,0bp,KAL,PM,30,1289.0939,0.000000,0.0426,0.000000,True
190,2025-12-10,0bp,PM,KAL,60,1782.4508,0.000000,0.0283,0.000000,True


In [ ]:
# Saving the dataframe in the defined directory 
# Ensure that you pick the directory corresponding to the event's data you have extracted in this example for 0bp cut 
final_results_df.to_csv("../../results/granger_causality/granger_causality_0bp_cut_2025.csv", index = False)